In [14]:
%load_ext autoreload
%autoreload

from methods.data_generation import DataGeneration
from methods.constants import NetworkSettings, DataGenerationSettings
from methods.methods_est import LinearRegressionEstimator

from glob import glob
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# TODO: 
- different network construction strategies (eg generate classes / grades)
- try Propensity Score 

In [15]:
# initialization 
network_settings = NetworkSettings()
dgp_settings = DataGenerationSettings()

In [16]:
data_generation = DataGeneration(
    n_nodes = network_settings.n_nodes,
    n_edges=network_settings.n_edges,
    n_features=dgp_settings.n_features,
    beta_mean=dgp_settings.beta_mean,
    beta_std=dgp_settings.beta_std,
    error_mean=dgp_settings.error_mean,
    error_std=dgp_settings.error_std,
    cov_mean_range=dgp_settings.cov_mean_range,
    cov_std_range=dgp_settings.cov_std_range,
    share_treatment=dgp_settings.share_treatment,
    n_influence_list=dgp_settings.n_influence_list,
    n_sim=dgp_settings.n_sim
)

In [17]:
assignment_types = dgp_settings.assignment_types
treatment_effects = dgp_settings.treatment_effects
network_types = network_settings.networks_types
neighbour_influences = dgp_settings.n_influence_list

In [18]:
def generate_assignments(data_generation, assignment_types, treatment_effects, network_configs, neighbour_influences, n_features, n_sim, write_results=False):
    results = []
    for assignment_type in assignment_types:
        data_generation.set_group_assignment_type(group_assignment_type=assignment_type)
        
        for treatment_mean, treatment_std in treatment_effects:
            data_generation.set_treatment_effect(treatment_effect_mean=treatment_mean, treatment_effect_std=treatment_std)
            
            for network_type, p_edges in network_configs:
                data_generation.set_network_type(network_type)
                if p_edges is not None:
                    data_generation.set_p_edges(p_edges)

                for influence in neighbour_influences:
                    data_generation.set_neighbour_influence(influence)
                    individ_data, dict_beta, dict_adj_matrix = data_generation.compute_neighbours_data_simulations(write_results=write_results)
                    estimator = LinearRegressionEstimator(
                            individ_data=individ_data,
                            n_features=n_features,
                            n_sim=n_sim,
                            network_type=network_type,
                            assignment_type=assignment_type,
                            neighbour_influence=influence,
                            treatment_mean=treatment_mean
                        )
                    estimator.calculate_results()
                    

In [19]:
generate_assignments(
    data_generation=data_generation,
    assignment_types=assignment_types, 
    treatment_effects=treatment_effects,
    network_configs=network_types,
    neighbour_influences=neighbour_influences,
    n_features=dgp_settings.n_features,
    n_sim=dgp_settings.n_sim,
)

100%|██████████| 100/100 [00:00<00:00, 350.64it/s]


In [20]:
def combine_pkl_results():
    pkl_files = glob("markdown_results/*.pkl") 
    dfs = [pd.read_pickle(file) for file in pkl_files]
    combined_df = pd.concat(dfs, ignore_index=True) 
    return combined_df

In [21]:
combine_pkl_results()

,Network,Assignment,Influence,Treatment,Mean Estimated,Mean True,Perc Diff (%)
0,watts_strogatz_graph,individual_covariates,0.3,0.9,1.123,0.9,22.046
1,erdos_renyi_graph,individual_covariates,0.9,0.6,0.433,0.6,32.321
2,barabasi_albert_graph,individual_and_neighbors,0.6,0.9,1.279,0.9,34.753
3,erdos_renyi_graph,random,0.9,0.9,0.858,0.9,4.754
4,barabasi_albert_graph,random,0.9,0.6,0.963,0.6,46.481
...,...,...,...,...,...,...,...
76,barabasi_albert_graph,individual_and_neighbors,0.6,0.6,0.743,0.6,21.262
77,erdos_renyi_graph,random,0.9,0.6,0.648,0.6,7.728
78,barabasi_albert_graph,random,0.9,0.9,1.201,0.9,28.655
79,barabasi_albert_graph,individual_and_neighbors,0.3,0.9,0.866,0.9,3.880
